# User Mapping Analysis

Created On: 7/25/2025

Created By: Juli Hirt

Objective of this notebook: 
* remove all instances of grey terms
* ensure all terms are 1:1 with their cleaned instances
* add a column for term stem


## Imports

In [1]:
import pandas as pd
import numpy as np 
import ast
import string

from nltk.stem import PorterStemmer
from IPython.display import display


stemmer = PorterStemmer()

In [2]:
import warnings
warnings.filterwarnings("ignore")

## Data Loads

In [3]:
df = pd.read_excel('Data/Inbound/all_terms_final - w_team_edits.xlsx', sheet_name='Raw Data')
df.head(3)

,metadata.ResponseId,metadata.Finished,metadata.Progress,metadata.TermTrack,metadata.QuestionId,QX_X.UserProvidedMoodTerm,QX_X.CleanedMoodTerm,QX_X.UserCategoryMapping,Rationale,QX.MultipleCategoryTerm,QX#1_X.MultipleCategoryMapping,QX#2_X_1.UserProvidedCategory,QX#2_X_1.CleanedCategory
0,R_1adu8XGo0BEYT2V,True,100,Avoid,Q10_1,Grind,grindy,No match,NaN,False,NaN,NaN,NaN
1,R_1adu8XGo0BEYT2V,True,100,Avoid,Q10_2,Boredom,boring,No match,NaN,False,NaN,NaN,NaN
2,R_1adu8XGo0BEYT2V,True,100,Favorite,Q6_1,Thilling,thrilling,Adventurous,NaN,True,"['Adventurous', 'Imaginative', 'Intense']",NaN,NaN


In [4]:
rm_df = pd.read_excel('Data/Inbound/all_terms_final - w_team_edits.xlsx', sheet_name='Removed Terms')
rm_df.head(3)

,QX_X.UserProvidedMoodTerm,QX_X.CleanedMoodTerm
0,Harvesty,harvesty
1,socializing,social
2,socializing,social


In [5]:
cln_df = pd.DataFrame(columns=df.columns)
cln_df.insert(7, 'QX_X.CleanedTermStem', '', True)
cln_df.head(3)

,metadata.ResponseId,metadata.Finished,metadata.Progress,metadata.TermTrack,metadata.QuestionId,QX_X.UserProvidedMoodTerm,QX_X.CleanedMoodTerm,QX_X.CleanedTermStem,QX_X.UserCategoryMapping,Rationale,QX.MultipleCategoryTerm,QX#1_X.MultipleCategoryMapping,QX#2_X_1.UserProvidedCategory,QX#2_X_1.CleanedCategory


In [6]:
rmf_df = pd.DataFrame(columns=df.columns)
rmf_df.head(3)

,metadata.ResponseId,metadata.Finished,metadata.Progress,metadata.TermTrack,metadata.QuestionId,QX_X.UserProvidedMoodTerm,QX_X.CleanedMoodTerm,QX_X.UserCategoryMapping,Rationale,QX.MultipleCategoryTerm,QX#1_X.MultipleCategoryMapping,QX#2_X_1.UserProvidedCategory,QX#2_X_1.CleanedCategory


## Cleaning

In [7]:
#duplicate the base dataframe to avoid modifying the original
temp_df = df.copy()
temp_df.insert(7, 'QX_X.CleanedTermStem', '', True)

# remove all whitespace from the term columns
temp_df['QX_X.UserProvidedMoodTerm'] = temp_df['QX_X.UserProvidedMoodTerm'].str.strip()
temp_df['QX_X.CleanedMoodTerm'] = temp_df['QX_X.CleanedMoodTerm'].str.strip()

In [8]:
# duplicate the base dataframe to avoid modifying the original
temp_rm_df = rm_df.copy()

#remove all whitespace from the term columns
temp_rm_df['QX_X.UserProvidedMoodTerm'] = temp_rm_df['QX_X.UserProvidedMoodTerm'].str.strip()   
temp_rm_df['QX_X.CleanedMoodTerm'] = temp_rm_df['QX_X.CleanedMoodTerm'].str.strip()

In [9]:
# Initialize variables for term cleaning
dic_arr = []
dic_temp = {
    'QX_X.UserProvidedMoodTerm': '',
    'QX_X.CleanedMoodTerm': '',
}
removed_term_count = 0
processed_row = 0

In [10]:
# set the mode for processing
mode = 'test'
#mode = 'prod'

In [11]:
for idx, row in temp_df.iterrows():
    user_provided_mood_term = row['QX_X.UserProvidedMoodTerm']
    cleaned_mood_term = row['QX_X.CleanedMoodTerm']
    
    # check if the term is in the removed terms
    if user_provided_mood_term in rm_df['QX_X.UserProvidedMoodTerm'].values or cleaned_mood_term in rm_df['QX_X.CleanedMoodTerm'].values:
        #for testing only
        if mode == 'test':
            print(f"Skipping removed term: {user_provided_mood_term} | {cleaned_mood_term}")
        removed_term_count += 1
        rmf_df = pd.concat([rmf_df, temp_df.iloc[[idx]]], ignore_index=True)

        continue

    # if the term is not in the removed terms, process it
    # 1 - check if the term is in the dictionary
    matched = False
    for dic in dic_arr:
        if user_provided_mood_term == dic['QX_X.UserProvidedMoodTerm'] or user_provided_mood_term.lower() == dic['QX_X.UserProvidedMoodTerm'].lower():
            matched = True
            # 2 - if it is, use the cleaned term from the dictionary
            cleaned_mood_term = dic['QX_X.CleanedMoodTerm']
            break
    if not matched:
        # 3 - if it is not, stem the term
        dic_temp_clone = dic_temp.copy()
        dic_temp_clone['QX_X.UserProvidedMoodTerm'] = user_provided_mood_term
        dic_temp_clone['QX_X.CleanedMoodTerm'] = cleaned_mood_term
        dic_arr.append(dic_temp_clone)

    # 4 - stem the cleaned term
    cleaned_mood_term_stem = stemmer.stem(cleaned_mood_term)

    # 5 - updated the dataframe
    temp_df.loc[idx, 'QX_X.CleanedMoodTerm'] = cleaned_mood_term
    temp_df.loc[idx, 'QX_X.CleanedTermStem'] = cleaned_mood_term_stem

    # 6 - add the cleaned term to the cleaned dataframe
    cln_df = pd.concat([cln_df, temp_df.iloc[[idx]]], ignore_index=True)
    processed_row += 1
    

if mode == 'test':
    print(f"Total removed terms: {removed_term_count}")


Skipping removed term: Whimsy | whimsical
Skipping removed term: cooperative | cooperative
Skipping removed term: tense | tense
Skipping removed term: Violence | violent
Skipping removed term: Harvesty | harvesty
Skipping removed term: Violent | violent
Skipping removed term: Tense | tense
Skipping removed term: Overwhelming | overwhelming
Skipping removed term: beautiful | beautiful
Skipping removed term: whimsical | whimsical
Skipping removed term: Beautiful | beautiful
Skipping removed term: socializing | social
Skipping removed term: socializing | social
Skipping removed term: Loud | overwhelming
Skipping removed term: Cliché | cliché
Skipping removed term: narrative | story-rich
Skipping removed term: colorful | whimsical
Skipping removed term: colorful | whimsical
Skipping removed term: puzzling | puzzling
Skipping removed term: Multi-tasking | overwhelming
Skipping removed term: Mapping | exploratory
Skipping removed term: Dungeon | exploratory
Skipping removed term: Drawbacks |

In [12]:
# must equal 1522
removed_term_count + processed_row 

1522

In [13]:
#remove extra columns
rmf_df.drop(columns=['QX_X.CleanedTermStem'], errors='ignore', inplace=True)

In [14]:
rmf_df.head(3)

,metadata.ResponseId,metadata.Finished,metadata.Progress,metadata.TermTrack,metadata.QuestionId,QX_X.UserProvidedMoodTerm,QX_X.CleanedMoodTerm,QX_X.UserCategoryMapping,Rationale,QX.MultipleCategoryTerm,QX#1_X.MultipleCategoryMapping,QX#2_X_1.UserProvidedCategory,QX#2_X_1.CleanedCategory
0,R_1FDENG6yljityVz,False,57,Favorite,Q6_7,Whimsy,whimsical,Peaceful,NaN,False,NaN,NaN,NaN
1,R_1hEHSUQxlm5DE33,True,100,Recent,Q2_2,cooperative,cooperative,No match,NaN,True,['Humorous'],NaN,NaN
2,R_1iPTsxjfKuMWHpi,True,100,Recent,Q2_6,tense,tense,Intense,NaN,False,NaN,NaN,NaN


In [15]:
cln_df.head(10)

,metadata.ResponseId,metadata.Finished,metadata.Progress,metadata.TermTrack,metadata.QuestionId,QX_X.UserProvidedMoodTerm,QX_X.CleanedMoodTerm,QX_X.CleanedTermStem,QX_X.UserCategoryMapping,Rationale,QX.MultipleCategoryTerm,QX#1_X.MultipleCategoryMapping,QX#2_X_1.UserProvidedCategory,QX#2_X_1.CleanedCategory
0,R_1adu8XGo0BEYT2V,True,100,Avoid,Q10_1,Grind,grindy,grindi,No match,NaN,False,NaN,NaN,NaN
1,R_1adu8XGo0BEYT2V,True,100,Avoid,Q10_2,Boredom,boring,bore,No match,NaN,False,NaN,NaN,NaN
2,R_1adu8XGo0BEYT2V,True,100,Favorite,Q6_1,Thilling,thrilling,thrill,Adventurous,NaN,True,"['Adventurous', 'Imaginative', 'Intense']",NaN,NaN
3,R_1adu8XGo0BEYT2V,True,100,Favorite,Q6_3,Funny,funny,funni,Humorous,NaN,False,NaN,NaN,NaN
4,R_1adu8XGo0BEYT2V,True,100,Favorite,Q6_4,Fulfilling,fulfilling,fulfil,Adventurous,NaN,True,"['Adventurous', 'Cute', 'Humorous', 'Sensual']",NaN,NaN
5,R_1adu8XGo0BEYT2V,True,100,Favorite,Q6_2,Exciting,exciting,excit,Intense,NaN,True,"['Adventurous', 'Intense']",NaN,NaN
6,R_1adu8XGo0BEYT2V,True,100,Recent,Q2_2,Noita - ominous,ominous,omin,Mysterious,NaN,True,"['Dark', 'Horror', 'Intense', 'Mysterious', 'S...",NaN,NaN
7,R_1adu8XGo0BEYT2V,True,100,Recent,Q2_4,Nine Sols - thrilling,thrilling,thrill,Intense,NaN,True,"['Adventurous', 'Aggressive', 'Cute', 'Imagina...",NaN,NaN
8,R_1adu8XGo0BEYT2V,True,100,Recent,Q2_3,Gloomwood - mysterious,mysterious,mysteri,Mysterious,NaN,False,NaN,NaN,NaN
9,R_1adu8XGo0BEYT2V,True,100,Recent,Q2_1,"Celeste - thrilling,",thrilling,thrill,Adventurous,NaN,True,"['Adventurous', 'Humorous', 'Intense', 'Sensual']",NaN,NaN


## Export Data

In [16]:
with pd.ExcelWriter('Data/Outbound/cleaned_terms.xlsx') as writer:
    cln_df.to_excel(writer, sheet_name='Cleaned Terms', index=False)
    rmf_df.to_excel(writer, sheet_name='Removed Terms', index=False)